# Retail Sales and Inventory Risk Analysis

## 1. Business Context and Analysis Scope

**Stakeholder:** Head of Commercial Controlling

### Business Problem

Management needs to understand whether current recorded inventory is aligned with recent product demand across stores.

Poor alignment may create two business risks:

- Insufficient stock for products with strong recent demand
- Excess stock for products with weak or declining demand

Missing inventory records may also prevent management from evaluating some store–product combinations correctly.

### Business Objective

Compare recent sales demand with current recorded inventory at the store–product level.

Using recent and preceding sales demand, current stock on hand, inventory-record status and estimated days of stock coverage, the analysis will identify:

- Potential replenishment candidates
- Possible excess-stock candidates
- Missing inventory records requiring data investigation

The analysis will provide investigation signals rather than automatic purchasing, inventory-transfer or promotional instructions.

### Analytical Questions

1. Which store–product combinations have confirmed zero stock or low stock coverage relative to recent demand and should be reviewed for replenishment?

2. Which store–product combinations have high stock coverage, declining demand or no recent demand and should be reviewed for inventory transfer or reduced future purchasing?

3. Which store–product combinations have missing inventory records that prevent reliable inventory decisions and require data investigation?

### Management Decisions Supported

The analysis will help management:

- prioritise replenishment reviews using recent demand, stock coverage and estimated gross profit;
- identify possible excess-stock cases using stock coverage, demand change and inventory value at cost;
- separate inventory-data issues from genuine zero-stock situations.

The results represent management-review candidates rather than automatic purchasing instructions.

### Main Measures

- Units sold during the latest 90-day period
- Units sold during the preceding 90-day period
- Percentage change in units sold between the two periods
- Average daily units sold during the latest 90-day period
- Current recorded stock on hand
- Estimated days of stock coverage
- Inventory-record status: positive stock, confirmed zero stock or missing inventory record

### Management KPIs

The final report will summarise:

- Number of confirmed zero-stock combinations with recent demand
- Number of positive-stock, low-coverage combinations with recent demand
- Number of positive-stock combinations with no recent demand
- Number of positive-stock, high-coverage combinations with declining demand
- Number of missing inventory combinations with recent demand

### Analytical Approach

The latest 90 days of available sales, from 3 July to 30 September 2023, will be used as the primary measure of recent demand.

The preceding equivalent period, from 4 April to 2 July 2023, will provide supporting context about whether demand increased or decreased.

Average daily demand will be calculated as:

`Units Sold During the Latest 90 Days ÷ 90`

The calculation will include all calendar days, including days without sales. Using only days with sales would overstate the normal daily demand rate.

Estimated days of stock coverage will be calculated as:

`Current Stock on Hand ÷ Average Daily Demand`

The following rules will be applied:

- Missing inventory records will remain classified as `inventory_record_missing`.
- Missing inventory records will be excluded from stock-coverage and inventory-risk classifications and reported separately as data-investigation candidates.
- Recorded zero stock combined with positive recent demand will produce zero days of stock coverage.
- Positive stock combined with zero recent demand will be classified separately because stock coverage cannot be meaningfully calculated.
- Positive stock combined with positive recent demand will produce an estimated stock-coverage value.
- Percentage demand change will not be calculated when the preceding period had zero sales because there is no valid comparison base.

Absolute stock quantities will not be used independently to classify inventory as low or high. The same quantity may be insufficient for a fast-selling product but excessive for a slow-selling product.

Stock will therefore be evaluated relative to recent demand through estimated days of stock coverage.

Low- and high-coverage thresholds will be configurable in the Tableau reporting layer. The initial dashboard defaults will classify seven days or fewer as low coverage and more than 180 days as high coverage.

The demand-decline threshold will also be configurable, with an initial default of a decrease of at least 50% compared with the preceding 90-day period.

These values are analyst-defined screening thresholds rather than confirmed operational stock targets. They can be adjusted in Tableau when additional business information becomes available.

For excess-stock review, the analysis will distinguish between:

- Combinations with no sales in either 90-day period
- Combinations with sales in the preceding period but no recent sales
- Combinations with positive recent sales but substantially declining demand and high stock coverage

### Data Limitations

The Inventory table does not contain an explicit snapshot date. The analysis assumes that the inventory snapshot approximately corresponds to the end of the sales period, 30 September 2023.

The dataset does not contain:

- Historical inventory levels
- Supplier lead times
- Delivery frequency
- Minimum order quantities
- Open purchase orders
- Safety-stock targets
- Target service levels

Therefore, the analysis cannot determine ideal stock quantities, reorder points or purchasing quantities.

The available sales period is insufficient for reliable long-term seasonality analysis. Time comparisons will therefore be limited to the two consecutive 90-day periods and interpreted as recent demand changes rather than forecasts.

A confirmed zero-stock record does not prove that sales were lost because the dataset does not show when the product became unavailable.

Final replenishment, purchasing, transfer and promotional decisions require additional operational information and consultation with the relevant business stakeholders.

## 2. Data Sources and Structure

The data dictionary is reviewed before analysing the individual tables.

**Dataset:** [Mexico Toy Sales — Maven Analytics](https://mavenanalytics.io/data-playground/mexico-toy-sales)

The project uses five analytical source tables: Products, Stores, Inventory, Sales and Calendar. The accompanying data dictionary describes the available fields.

In [62]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"

csv_files = sorted(raw_data_dir.glob("*.csv"))
[file.name for file in csv_files]

['calendar.csv',
 'data_dictionary.csv',
 'inventory.csv',
 'products.csv',
 'sales.csv',
 'stores.csv']

In [63]:
data_dictionary = pd.read_csv(
    raw_data_dir / "data_dictionary.csv"
)

data_dictionary


,Table,Field,Description
0,Products,Product_ID,Product ID
1,Products,Product_Name,Product name
2,Products,Product_Category,Product Category
3,Products,Product_Cost,Product cost ($USD)
4,Products,Product_Price,Product retail price ($USD)
5,Inventory,Store_ID,Store ID
6,Inventory,Product_ID,Product ID
7,Inventory,Stock_On_Hand,Stock quantity of the product in the store (in...
8,Stores,Store_ID,Store ID
9,Stores,Store_Name,Store name


## 3. Data Audit and Preparation
Each source table is inspected before any cleaning or transformation.
### 3.1 Products Table

#### Raw Inspection

The raw Products table is inspected to determine its grain, dimensions, data types and potential data-quality problems before applying transformations.

In [64]:
products = pd.read_csv(raw_data_dir / "products.csv")

products.head()


,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price
0,1,Action Figure,Toys,$9.99,$15.99
1,2,Animal Figures,Toys,$9.99,$12.99
2,3,Barrel O' Slime,Art & Crafts,$1.99,$3.99
3,4,Chutes & Ladders,Games,$9.99,$12.99
4,5,Classic Dominoes,Games,$7.99,$9.99


In [65]:
products.shape

(35, 5)

In [66]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Product_ID        35 non-null     int64
 1   Product_Name      35 non-null     str  
 2   Product_Category  35 non-null     str  
 3   Product_Cost      35 non-null     str  
 4   Product_Price     35 non-null     str  
dtypes: int64(1), str(4)
memory usage: 1.5 KB


Each row represents one product, uniquely identified by `Product_ID`. Table contains the product ID, product name, category, cost and selling price.

In [67]:
product_raw_checks = {
    "duplicate_product_ids": (products["Product_ID"].duplicated().sum()),
    "duplicate_complete_rows": (products.duplicated().sum()),
    "total_missing_values": (products.isna().sum().sum())
}

product_raw_checks

{'duplicate_product_ids': np.int64(0),
 'duplicate_complete_rows': np.int64(0),
 'total_missing_values': np.int64(0)}

In [68]:
products["Product_Category"].value_counts(dropna=False)

Product_Category
Toys                 9
Art & Crafts         8
Games                8
Sports & Outdoors    7
Electronics          3
Name: count, dtype: int64

#### Cleaning and Transformation

The initial inspection showed that `Product_Cost` and `Product_Price` were stored as text because they contained dollar signs and extra spaces. A separate clean DataFrame is created to preserve the original imported data.

In [69]:
products_clean = products.copy()

currency_columns = ["Product_Cost", "Product_Price"]

for column in currency_columns:
    products_clean[column] = (
        products_clean[column]
        .str.replace("$", "", regex=False)
        .str.strip()
        .astype(float)
    )

#### Post-cleaning Validation

The cleaned table is validated to confirm that the transformations produced appropriate data types and did not introduce missing or invalid values.

In [70]:
products_clean.dtypes

Product_ID            int64
Product_Name            str
Product_Category        str
Product_Cost        float64
Product_Price       float64
dtype: object

In [71]:
product_quality_checks = {
    "duplicate_product_ids": (products_clean["Product_ID"].duplicated().sum()),
    "total_missing_values": (products_clean.isna().sum().sum()),
    "non_positive_costs": ((products_clean["Product_Cost"] <= 0).sum()),
    "non_positive_prices": ((products_clean["Product_Price"] <= 0).sum()),
    "cost_greater_than_or_equal_to_price": ((products_clean["Product_Cost"]
                                             >= products_clean["Product_Price"]).sum())
}

product_quality_checks

{'duplicate_product_ids': np.int64(0),
 'total_missing_values': np.int64(0),
 'non_positive_costs': np.int64(0),
 'non_positive_prices': np.int64(0),
 'cost_greater_than_or_equal_to_price': np.int64(0)}

In [72]:
products_clean[
    ["Product_Cost", "Product_Price"]
].agg(["min", "max"])

,Product_Cost,Product_Price
min,1.99,2.99
max,34.99,39.99


#### Products Table Conclusion

The Products table contains 35 rows and 5 columns. Each row represents one unique product, identified by `Product_ID`, and includes its name, category, cost and selling price.

The table contains no missing values, duplicate rows or duplicate Product IDs. `Product_Category` contains five distinct, consistently formatted categories, with no missing or unexpected values.

The `Product_Cost` and `Product_Price` columns were transformed from text into floating-point numbers after removing dollar signs and extra spaces.

All post-cleaning validation checks passed. Neither costs nor selling prices contain zero or negative values, and every product's selling price is higher than its cost. Selling prices range from $2.99 to $39.99, while product costs range from $1.99 to $34.99.

Based on these checks, the Products table is ready for analysis.

### 3.2 Stores Table

#### Raw Inspection

The raw Stores table is inspected to determine its grain, dimensions, data types and potential data-quality problems before applying transformations.

In [73]:
stores = pd.read_csv(raw_data_dir / "stores.csv")

stores.head()


,Store_ID,Store_Name,Store_City,Store_Location,Store_Open_Date
0,1,Maven Toys Guadalajara 1,Guadalajara,Residential,1992-09-18
1,2,Maven Toys Monterrey 1,Monterrey,Residential,1995-04-27
2,3,Maven Toys Guadalajara 2,Guadalajara,Commercial,1999-12-27
3,4,Maven Toys Saltillo 1,Saltillo,Downtown,2000-01-01
4,5,Maven Toys La Paz 1,La Paz,Downtown,2001-05-31


In [74]:
stores.shape

(50, 5)

In [75]:
stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Store_ID         50 non-null     int64
 1   Store_Name       50 non-null     str  
 2   Store_City       50 non-null     str  
 3   Store_Location   50 non-null     str  
 4   Store_Open_Date  50 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.1 KB


Each row represents one store, uniquely identified by `Store_ID`. The table contains the store name, city, location type and opening date.

In [76]:
store_raw_checks = {
    "duplicate_store_ids": (stores["Store_ID"].duplicated().sum()),
    "duplicate_complete_rows": (stores.duplicated().sum()),
    "duplicate_store_names": (stores["Store_Name"].duplicated().sum()),
    "total_missing_values": (stores.isna().sum().sum())
}

store_raw_checks

{'duplicate_store_ids': np.int64(0),
 'duplicate_complete_rows': np.int64(0),
 'duplicate_store_names': np.int64(0),
 'total_missing_values': np.int64(0)}

In [77]:
stores["Store_Location"].value_counts(dropna=False)

Store_Location
Downtown       29
Commercial     12
Residential     6
Airport         3
Name: count, dtype: int64

In [78]:
stores["Store_City"].value_counts(dropna=False)

Store_City
Guadalajara         4
Monterrey           4
Cuidad de Mexico    4
Guanajuato          3
Puebla              3
Hermosillo          3
Saltillo            2
Mexicali            2
Campeche            2
Toluca              2
Chihuahua           2
Xalapa              2
La Paz              1
Pachuca             1
Cuernavaca          1
Chetumal            1
Tuxtla Gutierrez    1
San Luis Potosi     1
Merida              1
Zacatecas           1
Santiago            1
Aguascalientes      1
Ciudad Victoria     1
Oaxaca              1
Villahermosa        1
Chilpancingo        1
Morelia             1
Durango             1
Culiacan            1
Name: count, dtype: int64

The raw table contains no missing values, duplicate rows, duplicate Store IDs or duplicate store names. `Store_Location` contains four consistently formatted categories.

The city inspection identified a suspected spelling error: `Cuidad de Mexico` should be reviewed and potentially corrected to `Ciudad de Mexico`. `Store_Open_Date` is stored as text and requires conversion to a datetime data type.

#### Cleaning and Transformation

A separate clean DataFrame is created to preserve the raw data. The suspected city-name spelling error is corrected, and `Store_Open_Date` is converted from text to a datetime data type.

In [79]:
stores_clean = stores.copy()

stores_clean["Store_City"] = (
    stores_clean["Store_City"]
    .replace({"Cuidad de Mexico": "Ciudad de Mexico"})
)

stores_clean["Store_Open_Date"] = pd.to_datetime(
    stores_clean["Store_Open_Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [80]:
stores_clean.head()

,Store_ID,Store_Name,Store_City,Store_Location,Store_Open_Date
0,1,Maven Toys Guadalajara 1,Guadalajara,Residential,1992-09-18
1,2,Maven Toys Monterrey 1,Monterrey,Residential,1995-04-27
2,3,Maven Toys Guadalajara 2,Guadalajara,Commercial,1999-12-27
3,4,Maven Toys Saltillo 1,Saltillo,Downtown,2000-01-01
4,5,Maven Toys La Paz 1,La Paz,Downtown,2001-05-31


In [81]:
stores_clean.dtypes

Store_ID                    int64
Store_Name                    str
Store_City                    str
Store_Location                str
Store_Open_Date    datetime64[us]
dtype: object

In [82]:
stores_clean["Store_Open_Date"].agg(["min", "max"])

min   1992-09-18
max   2016-05-18
Name: Store_Open_Date, dtype: datetime64[us]

In [83]:
stores_clean.isna().sum()

Store_ID           0
Store_Name         0
Store_City         0
Store_Location     0
Store_Open_Date    0
dtype: int64

In [84]:
(
    stores_clean["Store_City"]
    == "Cuidad de Mexico"
).sum()

np.int64(0)

In [85]:
(
    stores_clean["Store_City"]
    == "Ciudad de Mexico"
).sum()

np.int64(4)

#### Stores Table Conclusion

The Stores table contains 50 rows and 5 columns. Each row represents one store, uniquely identified by `Store_ID`, and includes its name, city, location type and opening date.

The raw table contained no missing values, duplicate rows, duplicate Store IDs or duplicate store names. `Store_Location` contains four consistently formatted categories: Downtown, Commercial, Residential and Airport.

The city inspection identified four records containing the misspelled value `Cuidad de Mexico`. These values were corrected to `Ciudad de Mexico`.

`Store_Open_Date` was converted from text to a datetime data type. No invalid or missing dates were introduced during the conversion. Store opening dates range from 18 September 1992 to 18 May 2016.

Based on these checks, the Stores table is ready to be integrated with the other tables. The cross-table audit confirmed that no sales were recorded before the corresponding store’s opening date.

### 3.3 Inventory Table

#### Raw Inspection

The raw Inventory table is inspected to determine its grain, dimensions, data types and potential data-quality problems.

In [86]:
inventory = pd.read_csv(raw_data_dir / "inventory.csv")

inventory.head()

,Store_ID,Product_ID,Stock_On_Hand
0,1,1,27
1,1,2,0
2,1,3,32
3,1,4,6
4,1,5,0


In [87]:
inventory.shape

(1593, 3)

In [88]:
inventory.info()

<class 'pandas.DataFrame'>
RangeIndex: 1593 entries, 0 to 1592
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Store_ID       1593 non-null   int64
 1   Product_ID     1593 non-null   int64
 2   Stock_On_Hand  1593 non-null   int64
dtypes: int64(3)
memory usage: 37.5 KB


Each row represents the current stock level of one product at one store. The combination of `Store_ID` and `Product_ID` identifies an individual inventory record.

In [89]:
inventory_checks = {
    "total_missing_values": (
        inventory.isna().sum().sum()
    ),
    "duplicate_complete_rows": (
        inventory.duplicated().sum()
    ),
    "duplicate_store_product_pairs": (
        inventory.duplicated(
            subset=["Store_ID", "Product_ID"]
        ).sum()
    ),
    "negative_stock_records": (
        (inventory["Stock_On_Hand"] < 0).sum()
    ),
    "zero_stock_records": (
        (inventory["Stock_On_Hand"] == 0).sum()
    ),
    "minimum_stock": (
        inventory["Stock_On_Hand"].min()
    ),
    "maximum_stock": (
        inventory["Stock_On_Hand"].max()
    ),
    "unknown_store_ids": (
        ~inventory["Store_ID"].isin(
            stores_clean["Store_ID"]
        )
    ).sum(),
    "unknown_product_ids": (
        ~inventory["Product_ID"].isin(
            products_clean["Product_ID"]
        )
    ).sum()
}

inventory_checks

{'total_missing_values': np.int64(0),
 'duplicate_complete_rows': np.int64(0),
 'duplicate_store_product_pairs': np.int64(0),
 'negative_stock_records': np.int64(0),
 'zero_stock_records': np.int64(77),
 'minimum_stock': np.int64(0),
 'maximum_stock': np.int64(139),
 'unknown_store_ids': np.int64(0),
 'unknown_product_ids': np.int64(0)}

#### Cleaning and Transformation

No cleaning transformations are required. A separate copy is created for use in later integration and analysis.

In [90]:
inventory_clean = inventory.copy()

#### Inventory Completeness

The Inventory table contains valid store and product identifiers, but this does not prove that every possible store–product combination has an inventory record.

To evaluate inventory coverage, a complete grid of all stores and products is created and compared with the available inventory records. Missing combinations are retained as `inventory_record_missing` and are not interpreted as zero stock.

In [91]:
store_ids = stores_clean[["Store_ID"]]
product_ids = products_clean[["Product_ID"]]

all_store_product_pairs = store_ids.merge(
    product_ids,
    how="cross"
)

all_store_product_pairs.shape

(1750, 2)

In [92]:
inventory_complete = all_store_product_pairs.merge(
    inventory_clean,
    on=["Store_ID", "Product_ID"],
    how="left"
)

inventory_complete.head()

,Store_ID,Product_ID,Stock_On_Hand
0,1,1,27.0
1,1,2,0.0
2,1,3,32.0
3,1,4,6.0
4,1,5,0.0


In [93]:
inventory_complete["Stock_On_Hand"].isna().sum()

np.int64(157)

In [94]:
inventory_complete["Inventory_Status"] = "in_stock"

In [95]:
inventory_complete.loc[
    inventory_complete["Stock_On_Hand"] == 0,
    "Inventory_Status"
] = "out_of_stock"

In [96]:
inventory_complete.loc[
    inventory_complete["Stock_On_Hand"].isna(),
    "Inventory_Status"
] = "inventory_record_missing"

In [97]:
inventory_complete["Stock_On_Hand"] = (
    inventory_complete["Stock_On_Hand"].astype("Int64")
)

inventory_complete.dtypes

Store_ID            int64
Product_ID          int64
Stock_On_Hand       Int64
Inventory_Status      str
dtype: object

In [98]:
inventory_complete["Inventory_Status"].value_counts()

Inventory_Status
in_stock                    1516
inventory_record_missing     157
out_of_stock                  77
Name: count, dtype: int64

#### Inventory Table Conclusion

The Inventory table contains 1,593 rows and 3 columns. Each row represents the recorded stock level for one product at one store, identified by the combination of `Store_ID` and `Product_ID`.

The recorded inventory data contains no missing values, duplicate rows, duplicate store–product combinations or negative stock values. Every Store ID and Product ID exists in the corresponding cleaned dimension table.

Recorded stock levels range from 0 to 139 units. There are 77 store–product combinations with explicitly recorded zero stock. These are classified as `out_of_stock` and will be investigated later alongside recent sales demand.

A comparison with all 1,750 possible store–product combinations identified 157 combinations without an inventory record. These combinations are classified as `inventory_record_missing`. They are kept separate from confirmed zero-stock records because the absence of a record does not prove that stock is zero.

No cleaning of the original inventory values was required. However, the expanded `inventory_complete` table, including the inventory status classification, will be used in the subsequent analysis to prevent missing inventory records from being incorrectly treated as zero stock.

### 3.4 Sales Table

#### Raw Inspection

The raw Sales table is inspected to determine its grain, dimensions, data types and potential data-quality problems before applying transformations.

In [99]:
sales = pd.read_csv(
    raw_data_dir / "sales.csv"
)

sales.head()

,Sale_ID,Date,Store_ID,Product_ID,Units
0,1,2022-01-01,24,4,1
1,2,2022-01-01,28,1,1
2,3,2022-01-01,6,8,1
3,4,2022-01-01,48,7,1
4,5,2022-01-01,44,18,1


In [100]:
sales.shape

(829262, 5)

In [101]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 829262 entries, 0 to 829261
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   Sale_ID     829262 non-null  int64
 1   Date        829262 non-null  str  
 2   Store_ID    829262 non-null  int64
 3   Product_ID  829262 non-null  int64
 4   Units       829262 non-null  int64
dtypes: int64(4), str(1)
memory usage: 31.6 MB


Each row represents a sales record for one product at one store on one date. `Sale_ID` is expected to uniquely identify each record.

The dataset does not contain an order or receipt identifier. Therefore, individual products purchased as part of the same customer transaction cannot be grouped into complete orders.

#### Raw Data Validation

The raw table is checked for missing values, complete-row duplicates, duplicate Sale IDs and invalid unit quantities.

In [102]:
sales_raw_checks = {
    "total_missing_values": (
        sales.isna().sum().sum()
    ),
    "duplicate_complete_rows": (
        sales.duplicated().sum()
    ),
    "duplicate_sale_ids": (
        sales["Sale_ID"].duplicated().sum()
    ),
    "non_positive_unit_records": (
        (sales["Units"] <= 0).sum()
    ),
    "minimum_units": (
        sales["Units"].min()
    ),
    "maximum_units": (
        sales["Units"].max()
    )
}

sales_raw_checks

{'total_missing_values': np.int64(0),
 'duplicate_complete_rows': np.int64(0),
 'duplicate_sale_ids': np.int64(0),
 'non_positive_unit_records': np.int64(0),
 'minimum_units': np.int64(1),
 'maximum_units': np.int64(30)}

#### Cleaning and Transformation

A separate clean DataFrame is created to preserve the raw data. The `Date` column is converted from text into a datetime data type. Invalid date values are converted to `NaT` so they can be identified during validation.

In [103]:
sales_clean = sales.copy()

sales_clean["Date"] = pd.to_datetime(
    sales_clean["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [104]:
sales_clean.dtypes

Sale_ID                int64
Date          datetime64[us]
Store_ID               int64
Product_ID             int64
Units                  int64
dtype: object

In [105]:
sales_clean["Date"].agg(["min", "max"])

min   2022-01-01
max   2023-09-30
Name: Date, dtype: datetime64[us]

#### Post-cleaning and Relationship Validation

The cleaned Sales table is checked for invalid dates introduced during conversion. `Store_ID` and `Product_ID` are validated against the cleaned Stores and Products tables to confirm referential integrity.

In [106]:
sales_relationship_checks = {
    "invalid_dates_after_conversion": (
        sales_clean["Date"].isna().sum()
    ),
    "unknown_store_ids": (
        ~sales_clean["Store_ID"].isin(
            stores_clean["Store_ID"]
        )
    ).sum(),
    "unknown_product_ids": (
        ~sales_clean["Product_ID"].isin(
            products_clean["Product_ID"]
        )
    ).sum()
}

sales_relationship_checks

{'invalid_dates_after_conversion': np.int64(0),
 'unknown_store_ids': np.int64(0),
 'unknown_product_ids': np.int64(0)}

In [107]:
daily_store_product_combinations = (
    sales_clean[
        ["Date", "Store_ID", "Product_ID"]
    ]
    .drop_duplicates()
    .shape[0]
)

daily_store_product_combinations

88291

#### Sales Table Conclusion

The Sales table contains 829,262 rows and 5 columns. Each row represents an individual sales record identified by a unique `Sale_ID`. It should not be interpreted as a complete customer transaction or shopping basket because the dataset does not contain an order, receipt or customer identifier. Multiple sales records can occur for the same product, store and date. Therefore, units will be aggregated to the daily store–product level before demand metrics are calculated.

The 829,262 individual sales records correspond to 88,291 distinct date–store–product combinations. Sales will therefore be aggregated before calculating daily demand.

The raw table contains no missing values, duplicate rows or duplicate Sale IDs. All sales records contain positive quantities, ranging from 1 to 30 units.

The `Date` column was converted from text to a datetime data type. No invalid dates were introduced during conversion. The sales period runs from 1 January 2022 to 30 September 2023.

Every Store ID and Product ID referenced in Sales exists in the corresponding cleaned dimension table. Based on these checks, the Sales table is ready for integration. Its dates will also be validated against the Calendar table after that table is inspected.





### 3.5 Calendar Table

#### Raw Inspection

The Calendar table is inspected to confirm that it contains one unique record for every date in the sales period.

In [108]:
calendar = pd.read_csv(
    raw_data_dir / "calendar.csv"
)

calendar.head()

,Date
0,1/1/2022
1,1/2/2022
2,1/3/2022
3,1/4/2022
4,1/5/2022


In [109]:
calendar.shape

(638, 1)

In [110]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 638 entries, 0 to 637
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    638 non-null    str  
dtypes: str(1)
memory usage: 5.1 KB


In [111]:
calendar_raw_checks = {
    "total_missing_values": (
        calendar.isna().sum().sum()
    ),
    "duplicate_dates": (
        calendar["Date"].duplicated().sum()
    )
}

calendar_raw_checks

{'total_missing_values': np.int64(0), 'duplicate_dates': np.int64(0)}

Each row represents one calendar date. The `Date` column is initially stored as text and requires conversion to a datetime data type.

#### Cleaning and Transformation

A separate clean DataFrame is created, and the `Date` column is converted from text to a datetime data type.

In [112]:
calendar_clean = calendar.copy()

calendar_clean["Date"] = pd.to_datetime(
    calendar_clean["Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

#### Post-cleaning and Relationship Validation

The cleaned Calendar table is checked for invalid dates, duplicates, missing days within the date range and Sales dates that are absent from Calendar.

In [113]:
complete_date_range = pd.date_range(
    start=calendar_clean["Date"].min(),
    end=calendar_clean["Date"].max(),
    freq="D"
)

missing_calendar_dates = complete_date_range.difference(
    calendar_clean["Date"]
)

In [114]:
calendar_checks = {
    "invalid_dates_after_conversion": (
        calendar_clean["Date"].isna().sum()
    ),
    "duplicate_dates": (
        calendar_clean["Date"].duplicated().sum()
    ),
    "missing_dates_within_range": (
        len(missing_calendar_dates)
    ),
    "sales_dates_missing_from_calendar": (
        ~sales_clean["Date"].isin(
            calendar_clean["Date"]
        )
    ).sum(),
    "minimum_date": (
        calendar_clean["Date"].min()
    ),
    "maximum_date": (
        calendar_clean["Date"].max()
    )
}

calendar_checks

{'invalid_dates_after_conversion': np.int64(0),
 'duplicate_dates': np.int64(0),
 'missing_dates_within_range': 0,
 'sales_dates_missing_from_calendar': np.int64(0),
 'minimum_date': Timestamp('2022-01-01 00:00:00'),
 'maximum_date': Timestamp('2023-09-30 00:00:00')}

#### Calendar Table Conclusion

The Calendar table contains 638 rows and 1 column. Each row represents one calendar date within the available sales period.

The raw table contained no missing values or duplicate dates. The `Date` column was converted from text into a datetime data type, and no invalid dates were introduced during conversion.

The Calendar covers the period from 1 January 2022 to 30 September 2023. It contains every date within this range without gaps, and every date referenced in the Sales table exists in Calendar.

Based on these checks, the Calendar table is ready for integration. Additional attributes such as year, quarter, month and day of the week will be created later for SQL analysis and Tableau reporting.

## 4. Cross-Table Audit Summary

This section validates relationships between the cleaned Sales, Stores and Inventory tables. It checks whether any sales occurred before a store opened and investigates missing inventory combinations associated with recent sales.

In [115]:
sales_with_opening_dates = sales_clean.merge(
    stores_clean[["Store_ID", "Store_Open_Date"]],
    on="Store_ID",
    how="left"
)

sales_before_store_opening = (
    sales_with_opening_dates["Date"]
    < sales_with_opening_dates["Store_Open_Date"]
).sum()

sales_before_store_opening

np.int64(0)

In [116]:
missing_inventory_pairs = inventory_complete.loc[
    inventory_complete["Inventory_Status"]
    == "inventory_record_missing",
    ["Store_ID", "Product_ID"]
]

recent_end_date = sales_clean["Date"].max()
recent_start_date = recent_end_date - pd.Timedelta(days=89)

recent_sales = sales_clean[
    sales_clean["Date"].between(
        recent_start_date,
        recent_end_date
    )
]

len(missing_inventory_pairs), recent_start_date, recent_end_date

(157, Timestamp('2023-07-03 00:00:00'), Timestamp('2023-09-30 00:00:00'))

In [117]:
sales_with_missing_inventory = recent_sales.merge(
    missing_inventory_pairs,
    on=["Store_ID", "Product_ID"],
    how="inner"
)

sales_with_missing_inventory.head()

,Sale_ID,Date,Store_ID,Product_ID,Units
0,701901,2023-07-03,30,16,1
1,702239,2023-07-03,29,16,1
2,702289,2023-07-03,30,16,1
3,702515,2023-07-03,30,16,1
4,702524,2023-07-03,30,16,1


In [118]:
sales_with_missing_inventory.shape

(356, 5)

In [119]:
missing_pairs_with_recent_sales = (
    sales_with_missing_inventory[
        ["Store_ID", "Product_ID"]
    ]
    .drop_duplicates()
)

number_of_missing_pairs_with_sales = len(
    missing_pairs_with_recent_sales
)

units_sold_with_missing_inventory = (
    sales_with_missing_inventory["Units"].sum()
)

number_of_missing_pairs_with_sales, units_sold_with_missing_inventory

(23, np.int64(371))

In [120]:
affected_product_ids = (
    missing_pairs_with_recent_sales[
        "Product_ID"
    ].unique()
)

products_clean.loc[
    products_clean["Product_ID"].isin(
        affected_product_ids
    ),
    ["Product_ID", "Product_Name", "Product_Category"]
]

,Product_ID,Product_Name,Product_Category
15,16,Jenga,Games


In [121]:
cross_table_checks = {
    "sales_before_store_opening": int(
        sales_before_store_opening
    ),
    "missing_inventory_combinations": int(
        len(missing_inventory_pairs)
    ),
    "missing_combinations_with_recent_sales": int(
        number_of_missing_pairs_with_sales
    ),
    "units_sold_with_missing_inventory": int(
        units_sold_with_missing_inventory
    ),
    "affected_products": (
        products_clean.loc[
            products_clean["Product_ID"].isin(
                affected_product_ids
            ),
            "Product_Name"
        ].tolist()
    )
}

cross_table_checks

{'sales_before_store_opening': 0,
 'missing_inventory_combinations': 157,
 'missing_combinations_with_recent_sales': 23,
 'units_sold_with_missing_inventory': 371,
 'affected_products': ['Jenga']}

### Cross-Table Audit Conclusion

No sales were recorded before the corresponding store’s opening date.

The inventory completeness check identified 157 store–product combinations without an inventory record. During the latest 90-day period, from 3 July to 30 September 2023, 23 of these combinations generated 371 units of sales. All 23 combinations relate to Jenga.

This represents a specific inventory-data anomaly. The sales records demonstrate that Jenga was active in these stores, but the dataset does not provide corresponding inventory records. Without additional business information, these records cannot be interpreted as either zero stock or positive stock.

Missing inventory records will therefore be excluded from days-of-coverage calculations and from relative low- or high-coverage classifications. They will be reported separately as data-investigation candidates.

The audit established the following rules for subsequent analysis:

- Sales will be aggregated to the date–store–product level before daily demand is calculated.
- Confirmed zero stock will remain separate from missing inventory records.
- Missing inventory records will not be converted to zero.
- Average daily demand will include all calendar days in the latest 90-day period.
- Revenue and gross profit will be presented as estimates based on the available product-level price and cost. 
- Absolute stock quantities will not be used to define low or high inventory. Stock will be evaluated relative to recent demand and compared primarily between stores carrying the same product.

## 5. Export Cleaned Data for SQL

The cleaned source tables are exported as CSV files for loading into PostgreSQL. SQL will be used to combine the tables, calculate analytical metrics and prepare the reporting dataset.

In [122]:
processed_data_dir = project_root / "data" / "processed"

products_clean.to_csv(
    processed_data_dir / "products_clean.csv",
    index=False
)

stores_clean.to_csv(
    processed_data_dir / "stores_clean.csv",
    index=False
)

inventory_clean.to_csv(
    processed_data_dir / "inventory_clean.csv",
    index=False
)

sales_clean.to_csv(
    processed_data_dir / "sales_clean.csv",
    index=False
)

calendar_clean.to_csv(
    processed_data_dir / "calendar_clean.csv",
    index=False
)